[![AWS SDK for pandas](_static/logo.png "AWS SDK for pandas")](https://github.com/aws/aws-sdk-pandas)

# 44 - Amazon Bedrock

[Amazon Bedrock](https://aws.amazon.com/bedrock/) batch inference is the cheapest way to run a foundation model over thousands or millions of inputs, but the workflow has several rough edges: serialise inputs as JSONL, upload to S3, submit a `CreateModelInvocationJob`, poll for completion, then read and flatten the per-input output files.

AWS SDK for pandas turns that into three DataFrame-shaped calls:

- `wr.bedrock.invoke_batch` — submit a batch inference job over a DataFrame of prompts; returns the job ARN.
- `wr.bedrock.describe_invoke_batch` — check the job status without blocking or raising.
- `wr.bedrock.read_invoke_batch` — read the results, joined back to your input DataFrame.

In [ ]:
import getpass

import pandas as pd

import awswrangler as wr

By default `invoke_batch` submits the job and returns immediately with the job ARN — batch jobs commonly run for hours, so blocking is opt-in. Pass `wait=True` to poll until the job finishes, or call `read_invoke_batch(job_arn)` later.

**Prerequisites:**

- A DataFrame with **at least 100 rows** (Bedrock's minimum records per job). For smaller workloads, use the on-demand `InvokeModel` API instead.
- An IAM role that Bedrock can assume to read the input and write the output. See [Create a service role for batch inference](https://docs.aws.amazon.com/bedrock/latest/userguide/batch-iam-sr.html).
- An S3 bucket the role can read and write.

In [ ]:
role_arn = getpass.getpass("Bedrock batch inference role ARN: ")
bucket = getpass.getpass("S3 bucket for batch inputs/outputs: ")

input_uri = f"s3://{bucket}/bedrock-batch/input/"
output_uri = f"s3://{bucket}/bedrock-batch/output/"

### Converse-style batch (default)

When you pass `prompt_column`, `invoke_batch` builds a Converse request body for each row automatically. Use `inference_config` to override decoding parameters.

The example below uses 100 rows (Bedrock's minimum). `invoke_batch` returns immediately with the job ARN — the job itself usually runs for tens of minutes to several hours.

In [ ]:
capitals = [
    "France", "Germany", "Japan", "Italy", "Spain", "Portugal", "Brazil", "Canada", "Mexico", "Egypt",
]
prompts = pd.DataFrame(
    {
        "id": [f"REC{i:04d}" for i in range(100)],
        "country": [capitals[i % len(capitals)] for i in range(100)],
    }
)
prompts["prompt"] = prompts["country"].apply(lambda c: f"What is the capital of {c}? Answer in one word.")

job_arn = wr.bedrock.invoke_batch(
    df=prompts,
    model_id="anthropic.claude-3-haiku-20240307-v1:0",
    role_arn=role_arn,
    prompt_column="prompt",
    record_id_column="id",
    inference_config={"maxTokens": 32, "temperature": 0.0},
    input_s3_uri=input_uri,
    output_s3_uri=output_uri,
)
job_arn

### Checking on the job — `wr.bedrock.describe_invoke_batch`

Persist the `job_arn` somewhere durable (a database, a Step Function state, an EventBridge target). To check progress without blocking — e.g. from a scheduled harvester — use `describe_invoke_batch`, a thin wrapper over `GetModelInvocationJob` that never raises on job state:

In [ ]:
wr.bedrock.describe_invoke_batch(job_arn)["status"]

### Harvesting results — `wr.bedrock.read_invoke_batch`

When you are ready to read the results — typically minutes or hours later, possibly from a different process — call `read_invoke_batch`. It locates the JSONL output on S3, parses it, and (optionally) merges back onto your input DataFrame so you keep your original columns. The output S3 location and invocation type are read from `GetModelInvocationJob` — you don't need to remember them.

If the job is still running, `read_invoke_batch` raises `wr.exceptions.JobNotFinished` rather than blocking (or returning something you'd have to None-check). Pass `wait=True` to poll until the job reaches a terminal state instead:

In [ ]:
results = wr.bedrock.read_invoke_batch(
    job_arn,
    df=prompts,
    record_id_column="id",
    wait=True,
)
results[["id", "country", "output_text"]]

What you get back:

- All your original columns are preserved (joined back via `recordId`).
- `record_id` — the value used as `recordId` in the JSONL.
- `model_output` — the raw model response (Converse or InvokeModel format).
- `output_text` — convenience column for Converse jobs, the assistant's text response.
- `error` — populated for failed records; `model_output` is `None` in that row.

The job ARN and final status are accessible via `results.attrs["job_arn"]` and `results.attrs["status"]`.

### InvokeModel-style batch (full request control)

When the default Converse request body isn't what you want — e.g. you need a system prompt, multi-turn input, structured output, or a model that does not yet support Converse — pass a pre-built `modelInput` per row via `model_input_column` and set `invocation_type="InvokeModel"`.

In [ ]:
def claude_body(question: str) -> dict:
    return {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 32,
        "system": "You are a terse assistant.",
        "messages": [{"role": "user", "content": [{"type": "text", "text": question}]}],
    }


rows = pd.DataFrame(
    {
        "id": [f"REC{i:04d}" for i in range(100)],
        "question": [f"Capital of {capitals[i % len(capitals)]}?" for i in range(100)],
    }
)
rows["body"] = rows["question"].apply(claude_body)

job_arn_invoke = wr.bedrock.invoke_batch(
    df=rows,
    model_id="anthropic.claude-3-haiku-20240307-v1:0",
    role_arn=role_arn,
    model_input_column="body",
    record_id_column="id",
    invocation_type="InvokeModel",
    input_s3_uri=input_uri,
    output_s3_uri=output_uri,
)
job_arn_invoke